# Série 4

## Exercice 1
Ecrivez un programme qui interprête une liste d'instructions en RPN (Reverse Polish Notation).
* Si l'instruction est un entier, ajoutez-la au sommet de la pile.
* Les opérations + - * / déclenchent un `pop()` des deux derniers nombres de la pile et un `push()` du résultat de l'opération entre ces deux nombres.

Exemples:

- ```1 4 - 3 *``` donne comme résultat ```-9```

- ```3 4 * 6 / 1 2 +``` donne comme résultat ```2 3```

Votre solution doit utiliser une pile que vous avez implémentée. Elle doit utiliser une liste (tableau) de taille fixe sans pour autant hériter de `List`. Il faut utiliser les fonctions (que vous devez implémenter) suivantes:

In [1]:
class MyStack:
    def __init__(self, size: int):
        if size < 1:
          print("Cannot create zero-capacity array stack")
          return
        self.pile = [None for i in range(size)]
        self.cap = size
        self.top = -1

    def print(self):
        print(self.pile[: self.top + 1])

    def size(self):
        return self.top + 1

    def is_empty(self):
        return self.top == -1

    def push(self, o: int):
        if self.top + 1 == self.cap:
            raise FullStackException("Pile pleine !")
        self.top += 1
        self.pile[self.top] = o

    def pop(self):
      if self.is_empty():
        raise VoidStackException("Pile vide !")
      val = self.pile[self.top]
      self.top -= 1
      return val

class Error(Exception):
    pass

class FullStackException(Error):
    pass

class VoidStackException(Error):
    pass


In [2]:
s = MyStack(3)
assert s.size() == 0
assert s.is_empty() == True
s.push(1)
s.push(4)
s.print()
assert s.size() == 2
assert s.is_empty() == False
assert s.pop() == 4
s.print()
assert s.size() == 1
s.push(3)
s.push(10)
assert s.size() == 3
s.print()
try:
    s.push(12)
    print("Erreur: FullStackException doit être levée durant cette opération")
except FullStackException:
    pass
except:
    print("Erreur: FullStackException doit être levée durant cette opération")
assert s.pop() == 10
assert s.pop() == 3
assert s.pop() == 1
assert s.size() == 0
assert s.is_empty() == True

s = MyStack(3)
s.push(5)
assert s.size() == 1
assert s.is_empty() == False
s.print()
assert s.pop() == 5
assert s.size() == 0
assert s.is_empty() == True
s.print()
try:
    s.pop()
    print("Erreur: VoidStackException doit être levée durant cette opération")
except VoidStackException:
    pass
except:
    print("Erreur: VoidStackException doit être levée durant cette opération")

[1, 4]
[1]
[1, 3, 10]
[5]
[]


In [3]:
from os.path import split

# Param:
# entry: String[]
def rpn(entry):
  a = 0
  b = 0
  tokens = entry
  def is_int(t): #vérifie si l'entrée est conforme cad : si bien entier
      return isinstance(t, int) or (isinstance(t, str) and t.lstrip("-").isdigit())

  stack = MyStack(len(tokens)) #nombres d'éléments que va considérer la pile [3,1,+...]
  for tok in tokens :
    if is_int(tok) :
      stack.push(int(tok))
    elif tok in {"+","-","*","/"} :
      try: # On vérifie que la pille est
        b = stack.pop()
        a = stack.pop()
      except VoidStackException:
            raise ValueError("Expression invalide : pas assez d'opérandes pour l’opérateur.")
      if tok == "+" :
        stack.push(a+b)
      elif tok == "-" :
        stack.push(a-b)
      elif tok == "*" :
        stack.push(a*b)
      elif tok == "/":
        stack.push(int(a/b))

    else : raise ValueError("caractère inconnu")
  return stack #On renvoie la pile

In [4]:
s = rpn(["1","4","-","3","*"])
s.print() # doit être "-9"
s = rpn(["3","4","*","6","/","1","2","+"])
s.print() # doit être "2 3"

[-9]
[2, 3]


### Explications

On utilise une pile MyStack , c'est une structure LIFO (Last In, First Out). comme vue dans le cour du 6 octobre. Elle est implémentée comme un tableau de taille fixe avec :
push(x) : ajoute un élément au sommet.
pop() : enlève et retourne l’élément du sommet.
print() : affiche le contenu du bas vers le haut (par ex. 2 3).
C'est une structure de donnée abstraite qui est définit par ses actions qui nous est utile pour traiter les expressions RPN

Quand au RPN
On créer d'abord une pile de taille égale au nombre de tokens
On parcourt la pile pour chaque token si c’est un entier (ex. "3", "-5") → on le convertit en int et on le push dans la pile.
Si c’est un opérateur (+, -, *, /) :
On vérifie que la pile contient au moins 2 opérandes à l'aide de 2 pop successifs (si un seul élément on retourne une erreur) d'abord l'opérande droite puis le gauche.
On applique l’opération.
On push le résultat.

pour finir on retourner la pile finale.

## Exercice 2
Implémentez et testez une liste doublement chaînée et son itérateur. Implémentez une classe pour la liste et une autre pour son itérateur. Si cela vous aide, vous pouvez utiliser la classe `Node` du cours.

In [5]:
class Node:
    previous_node = None
    current_node = None
    next_node = None
    def __init__(self, previous_node = None, current_node = None, next_node = None):
        self.current_node = current_node
        self.previous_node = previous_node
        self.next_node = next_node

    def get(self):
        return self.current_node

    def set(self, node):
        self.current_node = node

    def get_next(self):
        return self.next_node

    def set_next(self, node):
        self.next_node = node

    def get_previous(self):
        return self.previous_node

    def set_previous(self, node):
        self.previous_node = node


class DoublyLinkedList:
    def __init__ (self):
        self._head = None
        self._tail = None

    # retourne une instance de l'itérateur DoublyLinkedIterator ayant le
    # premier élément de la liste comme position initiale.
    def begin(self):
        if self._head is None:
            return None
        return DoublyLinkedIterator(self, self._head)

    # retourne une instance de l'itérateur DoublyLinkedIterator ayant le
    # dernier élément de la liste comme position initiale
    def end(self):
        if self._tail is None:
            return None
        return DoublyLinkedIterator(self, self._tail)

    # ajoute un élément à la fin de la liste
    def add(self, e):
        new_node = Node(self._tail, e, None)
        if self._tail is None:
            self._head = new_node
            self._tail = new_node
        else:
            self._tail.set_next(new_node)
            self._tail = new_node

    # supprime l'élément en tête de liste (le premier) et le retourne
    def remove(self):
        if self._head is None:
            return None
        else:
            e = self._head.get()
            self._head = self._head.get_next()
            if self._head is not None:
                self._head.set_previous(None)
            else:
                self._tail = None
            return e # retourne l'élément supprimé

    # retourne une valeur booléenne, selon que la liste est vide ou non
    def is_empty(self):
        if self._head is None:
            return True
        else:
            return False


class DoublyLinkedIterator:
    def __init__(self, dll=None, node=None):
        self._list = dll
        self._current = node

    # modifie la valeur de l'élément actuellement itéré
    def set(self,e):
        self._current.set(e)

    # retourne la valeur stockée dans l'élément actuellement itéré
    def get(self):
        return self._current.get()

    # retourne une instance de l'itérateur sur le prochain élément de la liste
    # s'il y en a un
    def increment(self):
        if self._current is None or self._current.get_next() is None:
            return None
        return DoublyLinkedIterator(self._list, self._current.get_next())

    # retourne une instance de l'itérateur sur l'élément précédent de la liste
    # s'il y en a un
    def decrement(self):
        if self._current is None or self._current.get_previous() is None:
            return None
        return DoublyLinkedIterator(self._list, self._current.get_previous())

    # retourne une valeur booléenne, selon si l'itérateur passé en paramètre énumère
    # la même liste et est positionner au même endroit.
    # Autrement dit, si les deux itérateurs sont sur le même élément
    def equals(self,o):
        if self._list is o._list and self._current is o._current: #is car doit pointer le mm élément
            return True
        else:
            return False

In [6]:
dll = DoublyLinkedList()
assert dll.is_empty() == True
dll.add(3)
assert dll.begin().get() == 3
assert dll.is_empty() == False
dll.add(10)
dll.add(9)
assert dll.end().get() == 9

it = dll.begin()
assert it.get() == 3
it = it.increment()
assert it.get() == 10
it = it.decrement()
assert it.equals(dll.begin()) == True
it.set(4)
assert it.get() == 4

assert dll.remove() == 4
assert dll.begin().get() == 10
assert dll.remove() == 10
assert dll.remove() == 9
assert dll.is_empty() == True

### Explications

La première classe Node (donné sur moodle) nous sert de base pour la liste doublement chainée. Chaque node de cette classe contiendra la valeure à stocker (current_node), la valeure précédente(previous_node) et la suivante(next_node). Les methodes présentent dans cette classe permettent d'ajouter (getter) ou de modifier (setter) les 3 éléments vus précédemment.

La classe DoublyLinkedList représente la liste en elle-même. Elle contient 2 attributs principaux : _head (premier élément de la liste) et _tail (dernier élément). A la création, ces deux valeurs sont None car la liste est vide. La methode add() permet d’ajouter des éléments à la fin de la liste. Si cette liste est vide, le nouvel élément devient à la fois le head et le tail. Sinon, il est relié au dernier élément et devient le nouveau tail. La méthode remove() supprime le premier élément de la liste(head) et retourne sa valeure. Si la liste devient vide, head et tail deviennent None. La méthode is_empty() vérifie si la liste est vide en regardant si le head = None. Les méthodes begin() et end() renvoient un itérateur positioné sur le premier élément(pour begin) et sur le dernier(pour end).

La classe DoublyLinkedIterator nous sert à parcourir la liste. Elle contient les attributs _list(la liste sur laquelle on travaille) et _current(qui pointe sur le node sélectionné). Les méthodes get() et set() permettent respectivement de retourner et de modifier le node que lequel on est. Les méthodes increment() et decrement() permettent de parcourir la liste(la première méthode avance d’un élément, la deuxième recule d’un élément). La méthode equals() regarde si 2 itérateurs pointent exactement sur le même node de la même liste.


## Exercice 3
Implémentez et testez une classe qui crée et rempli la liste doublement chaînée créée dans l'exercice 2 avec _n_ nombres aléatoires entre _0_ et _2n_. Utilisez l'implémentation de l'itérateur de l'exercice 2 pour:
* itérer dans la liste et afficher chaque élément
* afficher le premier élément de la liste et la position de la première occurence de X dans la liste:
  *  le cas où X n’est pas présent dans la liste doit être géré avec une exception dédiée `ItemNotFound`
* itérer la liste à l'envers (du dernier au premier élément) et afficher chaque élément dans l'ordre d'itération
* retirer chaque occurrence de X de la liste (elle peut contenir des éléments à double)
  * le cas où X n’est pas présent dans la liste doit être géré avec une exception dédiée `ItemNotFound`

In [7]:
import random

class ItemNotFound(Exception):
    pass

class TestLinkedList:
    list = None

    def __init__(self, n):
        self.list= DoublyLinkedList() #crée une list

    def fill_list(self,n):
        for i in range(n):
            self.list.add(random.randint(0,2*n)) # remplit la liste entièrement aléatoirement entre 0 et n*2

    def print_list_forwards(self):
        pointer=self.list.begin() #met le pointeur au début de la liste
        while pointer is not None: #tant que le pointeur n'est pas vide
            print(pointer.get())  # affiche l'élément
            pointer=pointer.increment() #passe au prochain pointeur


    def first_element(self):
        if self.list.is_empty():  #vérifie que la liste n'est pas vide
          raise ItemNotFound("List is empty")
        print("Head of the list", self.list._head.get()) #affiche la tête de la liste


    def first_occurence(self, x):
        pointer = self.list.begin()
        position = 0
        while pointer is not None:
            if pointer.get() == x:  #si le pointeur est égal a x
                print(position, x)
                return #alors on affiche la position et x
            pointer = pointer.increment() #on passe au prochain
            position += 1 #augmente la postion
        raise ItemNotFound("Item not found")

    def print_list_backwards(self):
        pointer=self.list.end() #même chose que print_list_forward mais on prend méthode end et decrement
        while pointer is not None:
            print(pointer.get())
            pointer=pointer.decrement()

    def remove_element(self, x):
        pointer=self.list.begin()
        found = False
        while pointer is not None:
            if pointer.get() == x: #on entre dans le if si x est trouvé
              current_node = pointer._current
              previous_node = current_node.get_previous() # On défini les noeuds current, next et previous
              next_node = current_node.get_next()

              if previous_node: #Si il y a un noeuds avant alors on le lie au noeud next
                previous_node.set_next(next_node)
              else:
                self.list._head = next_node # si le noeud n'existe pas alors c'est qu'on est en tête de liste et on passe la tête au next_node

              if next_node: #meme chose que previous mais cette fois on regarde si il y a un noeud apres et on le lie au noeud previous
                next_node.set_previous(previous_node)
              else:
                self.list._tail = previous_node #Si pas de noeud après alors on est en fin de liste alors on met previous_node comme nouveau tail

              found = True #pour dire si on en trouve au moins 1
            pointer=pointer.increment()
        if not found:
          raise ItemNotFound("Item not found")

In [8]:
# Quelques tests à titre indicatif
n = 5
test = TestLinkedList(n)
test.print_list_forwards()
print()
test.fill_list(n)
test.first_element()
print()
print("Must be the first element of the list (index = 0)")
test.first_occurence(test.list.begin().get())
test.first_occurence(test.list.end().get())

try:
    test.first_occurence(n*2+1)
    print("Error: ItemNotFound exception must be raised")
except ItemNotFound:
    pass
except:
    print("Error: ItemNotFound exception must be raised")

test.print_list_backwards()
print()
test.remove_element(test.list.begin().get())
test.print_list_forwards()
test.remove_element(test.list.end().get())
test.print_list_forwards()

try:
    test.remove_element(n*2+1)
    print("Error: ItemNotFound exception must be raised")
except ItemNotFound:
    pass
except:
    print("Error: ItemNotFound exception must be raised")


Head of the list 8

Must be the first element of the list (index = 0)
0 8
0 8
8
6
10
6
8

6
10
6
10


### Explications

La méthode remove_element ne suprimme pas x mais prend les liens avant et après lui et les lies puis il disparrait grâce au ramasse-miette de Python

Toutes les autres explications sont dans le code.